In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from gensim.models import Word2Vec

# 1. Cargar datos
df = pd.read_csv("../Datos/Preprocesados/tcga_preprocesado.csv")
y = df['t']

# IMPORTANTE: Pon aquí la columna que ganó tu torneo de preprocesamiento
X_texto = df['text']

# 2. Word2Vec necesita listas de palabras, no frases enteras. 
# Convertimos cada informe en una lista de tokens.
X_tokens = X_texto.apply(lambda x: str(x).split())

# 3. Split (Hacemos el split ANTES de entrenar Word2Vec para no hacer trampa)
X_train_tokens, X_test_tokens, y_train, y_test = train_test_split(
    X_tokens, y, test_size=0.20, random_state=42, stratify=y
)

print("🧠 Entrenando modelo Word2Vec médico desde cero...")

# 4. Entrenar Word2Vec solo con el Train
# vector_size=100 (cada palabra será un vector de 100 dimensiones)
# window=5 (mira 5 palabras a la izquierda y 5 a la derecha para entender el contexto)
# min_count=2 (ignora palabras que salen menos de 2 veces)
w2v_model = Word2Vec(sentences=X_train_tokens, vector_size=100, window=5, min_count=2, workers=4)

# 5. Función para convertir un informe entero en un solo vector
# (Hacemos la media de los vectores de todas las palabras de ese informe)
def document_vector(doc_tokens, model):
    # Filtramos palabras que el modelo conoce
    palabras_conocidas = [word for word in doc_tokens if word in model.wv.key_to_index]
    if len(palabras_conocidas) == 0:
        return np.zeros(model.vector_size)
    # Calculamos la media de los vectores de esas palabras
    return np.mean(model.wv[palabras_conocidas], axis=0)

print("🔄 Convirtiendo textos a vectores promediados...")

# 6. Transformar Train y Test a vectores numéricos
X_train_w2v = np.array([document_vector(tokens, w2v_model) for tokens in X_train_tokens])
X_test_w2v = np.array([document_vector(tokens, w2v_model) for tokens in X_test_tokens])

# 7. Entrenar y evaluar con un modelo clásico (Regresión Logística para comparar)
print("🚀 Entrenando clasificador sobre Embeddings...")
modelo_lr = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
modelo_lr.fit(X_train_w2v, y_train)

score_w2v = f1_score(y_test, modelo_lr.predict(X_test_w2v), average='macro')

print("-" * 50)
print(f"📊 Macro-F1 con Word2Vec (Propios) : {score_w2v:.4f}")
print("-" * 50)

🧠 Entrenando modelo Word2Vec médico desde cero...
🔄 Convirtiendo textos a vectores promediados...
🚀 Entrenando clasificador sobre Embeddings...
--------------------------------------------------
📊 Macro-F1 con Word2Vec (Propios) : 0.4453
--------------------------------------------------


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, LSTM
from tensorflow.keras import backend as K

# 1. Carga de datos y preparación de etiquetas
df = pd.read_csv("../Datos/Preprocesados/tcga_preprocesado.csv")

mapeo_etiquetas = {'T1': 0, 'T2': 1, 'T3': 2, 'T4': 3}
y = df['t'].map(mapeo_etiquetas)

columnas_a_probar = ['text', 'text_basico', 'text_lema', 'text_pos', 'text_stem_nltk', 'text_lema_nltk']
resultados_dl = []

vocab_size = 5000
max_length = 300 

print("Iniciando evaluación de Redes Neuronales (CNN y LSTM)...")

for col in columnas_a_probar:
    print(f"\nEntrenando modelos para la columna: {col}...")
    
    # Aseguramos que los datos sean strings para evitar errores del Tokenizer
    X = df[col].astype(str)
    
    # 2. División Train/Test (dentro del bucle porque X cambia)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    
    # 3. Tokenización y Padding
    tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train)
    
    X_train_seq = tokenizer.texts_to_sequences(X_train)
    X_test_seq = tokenizer.texts_to_sequences(X_test)
    
    X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
    X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')
    
    # 4. Pesos de clases
    pesos_clases = compute_class_weight(
        class_weight='balanced', classes=np.unique(y_train), y=y_train
    )
    dict_pesos = dict(enumerate(pesos_clases))

    # --- LIMPIEZA DE SESIÓN DE KERAS ---
    # Crucial al iterar modelos para evitar que la memoria RAM/GPU se llene
    K.clear_session()
    
    # ==========================================
    # 5. Construcción y Entrenamiento de la CNN
    # ==========================================
    modelo_cnn = Sequential([
        Embedding(input_dim=vocab_size, output_dim=100, input_length=max_length),
        Conv1D(filters=128, kernel_size=3, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(4, activation='softmax')
    ])
    
    modelo_cnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    # Entrenamos silenciosamente (verbose=0)
    modelo_cnn.fit(X_train_pad, y_train, epochs=10, batch_size=32, 
                   validation_split=0.1, class_weight=dict_pesos, verbose=0)
    
    y_pred_clases_cnn = np.argmax(modelo_cnn.predict(X_test_pad, verbose=0), axis=1)
    score_cnn = f1_score(y_test, y_pred_clases_cnn, average='macro')

    # ==========================================
    # 6. Construcción y Entrenamiento de la LSTM
    # ==========================================
    modelo_lstm = Sequential([
        Embedding(input_dim=vocab_size, output_dim=100, input_length=max_length),
        LSTM(64, dropout=0.2, recurrent_dropout=0.2),
        Dense(32, activation='relu'),
        Dropout(0.5),
        Dense(4, activation='softmax')
    ])
    
    modelo_lstm.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    modelo_lstm.fit(X_train_pad, y_train, epochs=10, batch_size=32, 
                    validation_split=0.1, class_weight=dict_pesos, verbose=0)
    
    y_pred_clases_lstm = np.argmax(modelo_lstm.predict(X_test_pad, verbose=0), axis=1)
    score_lstm = f1_score(y_test, y_pred_clases_lstm, average='macro')
    
    # ==========================================
    # 7. Guardar resultados
    # ==========================================
    resultados_dl.append({
        'Preprocesamiento': col, 
        'Macro-F1 (CNN)': score_cnn,
        'Macro-F1 (LSTM)': score_lstm
    })

# 8. Presentación de resultados finales
df_resultados_dl = pd.DataFrame(resultados_dl).sort_values(by='Macro-F1 (CNN)', ascending=False)

print("\n" + "="*60)
print(" RESULTADOS FINALES DE DEEP LEARNING (Macro-F1)")
print("="*60)
print(df_resultados_dl.to_string(index=False))

Iniciando evaluación de Redes Neuronales (CNN y LSTM)...

Entrenando modelos para la columna: text...



c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Entrenando modelos para la columna: text_basico...


c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Entrenando modelos para la columna: text_lema...


...


Entrenando modelos para la columna: text_stem_nltk...


c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Entrenando modelos para la columna: text_lema_nltk...


c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



 RESULTADOS FINALES DE DEEP LEARNING (Macro-F1)
Preprocesamiento  Macro-F1 (CNN)  Macro-F1 (LSTM)
            text        0.717550         0.314570
       text_lema        0.626806         0.277238
  text_stem_nltk        0.624580         0.304474
        text_pos        0.623142         0.250291
     text_basico        0.616879         0.271169
  text_lema_nltk        0.616381         0.292400
